In [1]:
import pandas as pd
from pathlib import Path

data_folder = Path(".")
csv_files = sorted(data_folder.glob("*.csv"))
print("Number of csv file found:",len(csv_files))
for file in csv_files:
    print(file.name)

                   
                   


Number of csv file found: 6
ratings.csv
reviews.csv
subscribers.csv
titles.csv
watch_history.csv
watchlist.csv


In [2]:
#loading all six files into Python

all_dfs = {}

for file in csv_files:
    table_name = file.stem
    all_dfs[table_name] = pd.read_csv(file)
    rows,columns = all_dfs[table_name].shape
    print(f"{file.name}:{rows}rows,{columns}columns")

ratings.csv:130000rows,5columns
reviews.csv:110000rows,7columns
subscribers.csv:15000rows,14columns
titles.csv:9000rows,21columns
watch_history.csv:650000rows,10columns
watchlist.csv:65000rows,5columns


In [3]:
#Creating the basic profile summary


profile_summary= []

for table_name,df in all_dfs.items():
    profile_summary.append({"Table Name":table_name,"Row Count":df.shape[0],"Column Count":df.shape[1]})

profile_summary_df = pd.DataFrame(profile_summary)
profile_summary_df

,Table Name,Row Count,Column Count
0,ratings,130000,5
1,reviews,110000,7
2,subscribers,15000,14
3,titles,9000,21
4,watch_history,650000,10
5,watchlist,65000,5


In [4]:
#Checking column names and data types

for table_name,df in all_dfs.items():
    print(f"\n----{table_name.upper()}----")
    print(df.dtypes)


----RATINGS----
rating_id         int64
subscriber_id    object
title_id         object
rating            int64
rating_date      object
dtype: object

----REVIEWS----
review_id         int64
subscriber_id    object
title_id         object
review_text      object
sentiment        object
helpful_votes     int64
review_date      object
dtype: object

----SUBSCRIBERS----
subscriber_id         object
signup_date           object
country               object
region                object
age                    int64
gender                object
plan_type             object
monthly_price_usd    float64
household_size         int64
primary_device        object
payment_method        object
tenure_months          int64
is_active               bool
churn_date            object
dtype: object

----TITLES----
title_id                 object
title_name               object
type                     object
primary_genre            object
country                  object
language                 object
r

In [5]:
#checking missing values


missing_summary = []

for table_name,df in all_dfs.items():
    missing_count = df.isnull().sum()
    missing_percentage = (missing_count/len(df)*100).round(2)

    table_missing = pd.DataFrame({
    "Table Name":table_name,"Column Name":df.columns,"Missing Count":missing_count,"Missing Percentage":missing_percentage})

    table_missing = table_missing[table_missing["Missing Count"]>0]

    missing_summary.append(table_missing)

missing_values_df = pd.concat(missing_summary,ignore_index=True)

missing_values_df

,Table Name,Column Name,Missing Count,Missing Percentage
0,subscribers,churn_date,11199,74.66
1,titles,license_expiry,1966,21.84


In [6]:
#checking exact duplicate rows

duplicate_summary = []

for table_name, df in all_dfs.items():
    duplicate_count = df.duplicated().sum()
    duplicate_percentage = (
        duplicate_count / len(df) * 100).round(2)

    duplicate_summary.append({
        "Table Name": table_name,
        "Duplicate Rows": duplicate_count,
        "Duplicate Percentage": duplicate_percentage})

duplicate_summary_df = pd.DataFrame(duplicate_summary)

duplicate_summary_df

,Table Name,Duplicate Rows,Duplicate Percentage
0,ratings,0,0.0
1,reviews,0,0.0
2,subscribers,0,0.0
3,titles,0,0.0
4,watch_history,0,0.0
5,watchlist,0,0.0


In [7]:
#Checking duplicate IDs


primary_keys = {
    "ratings": "rating_id",
    "reviews": "review_id",
    "subscribers": "subscriber_id",
    "titles": "title_id",
    "watch_history": "watch_id",
    "watchlist": "watchlist_id"}

key_duplicate_summary = []

for table_name, key_column in primary_keys.items():
    df = all_dfs[table_name]

    duplicate_mask = df[key_column].duplicated(keep=False)
    affected_rows = duplicate_mask.sum()
    duplicate_ids = df.loc[
        duplicate_mask, key_column].nunique()

    key_duplicate_summary.append({
        "Table Name": table_name,
        "ID Column": key_column,
        "Duplicate ID Values": duplicate_ids,
        "Affected Rows": affected_rows,
        "Affected Percentage": round(
            affected_rows / len(df) * 100, 2)})

key_duplicates_df = pd.DataFrame(key_duplicate_summary)

key_duplicates_df

,Table Name,ID Column,Duplicate ID Values,Affected Rows,Affected Percentage
0,ratings,rating_id,0,0,0.0
1,reviews,review_id,0,0,0.0
2,subscribers,subscriber_id,0,0,0.0
3,titles,title_id,0,0,0.0
4,watch_history,watch_id,0,0,0.0
5,watchlist,watchlist_id,0,0,0.0


#Exact duplicate rows: 0
#Duplicate primary ID values: 0
#Affected rows: 0

#This means all six primary-key columns are unique.

In [8]:
#Inspect date formats

date_columns = {
    "ratings": ["rating_date"],
    "reviews": ["review_date"],
    "subscribers": ["signup_date", "churn_date"],
    "titles": ["date_added", "license_expiry"],
    "watch_history": ["watch_date"],
    "watchlist": ["added_date"]}

for table_name, columns in date_columns.items():
    print(f"\n--- {table_name.upper()} ---")

    for column in columns:
        sample_values = (
            all_dfs[table_name][column]
            .dropna()
            .astype(str)
            .head(5)
            .tolist())

        print(f"{column}: {sample_values}")


--- RATINGS ---
rating_date: ['2026-02-06', '2025-03-21', '2024-05-26', '2026-05-22', '2026-05-16']

--- REVIEWS ---
review_date: ['2023-11-04', '2026-04-25', '2024-07-31', '2022-12-13', '2025-10-09']

--- SUBSCRIBERS ---
signup_date: ['2023-01-09', '2020-02-05', '2022-04-15', '2024-10-10', '2020-04-12']
churn_date: ['2022-12-08', '2021-12-15', '2024-10-19', '2020-07-22', '2026-05-08']

--- TITLES ---
date_added: ['2026-04-05', '2026-05-01', '2026-04-17', '2018-12-08', '2016-05-11']
license_expiry: ['2028-08-03', '2027-02-16', '2029-10-11', '2028-11-21', '2029-04-30']

--- WATCH_HISTORY ---
watch_date: ['2024-01-01', '2025-10-18', '2026-04-26', '2026-03-09', '2023-12-27']

--- WATCHLIST ---
added_date: ['2023-06-05', '2023-01-31', '2022-06-21', '2023-12-23', '2022-02-02']


In [9]:
#All eight date columns use the consistent format YYYY-MM-DD. We can safely convert them to pandas datetime.

In [10]:
#Convert and validate date columns

date_conversion_summary = []

for table_name, columns in date_columns.items():
    for column in columns:
        original_values = all_dfs[table_name][column].copy()

        converted_values = pd.to_datetime(
            original_values,
            format="%Y-%m-%d",
            errors="coerce")

        invalid_count = (
            original_values.notna() &
            converted_values.isna()).sum()

        all_dfs[table_name][column] = converted_values

        date_conversion_summary.append({
            "Table Name": table_name,
            "Date Column": column,
            "Invalid Dates": invalid_count,
            "New Data Type": str(
                all_dfs[table_name][column].dtype)})

date_conversion_df = pd.DataFrame(
    date_conversion_summary)

date_conversion_df

,Table Name,Date Column,Invalid Dates,New Data Type
0,ratings,rating_date,0,datetime64[ns]
1,reviews,review_date,0,datetime64[ns]
2,subscribers,signup_date,0,datetime64[ns]
3,subscribers,churn_date,0,datetime64[ns]
4,titles,date_added,0,datetime64[ns]
5,titles,license_expiry,0,datetime64[ns]
6,watch_history,watch_date,0,datetime64[ns]
7,watchlist,added_date,0,datetime64[ns]


All eight date columns were converted to datetime64[ns].
Invalid dates found: 0
Price and duration columns were already numeric.

In [11]:
#Flag invalid watch durations


#Checking whether any viewing session is longer than the content itself.


watch_history_df = all_dfs["watch_history"]

duration_outlier_mask = (
    watch_history_df["watch_duration_min"] >
    watch_history_df["content_duration_min"])

watch_history_df["duration_outlier_flag"] = (
    duration_outlier_mask)

duration_outlier_count = duration_outlier_mask.sum()

duration_outlier_percentage = round(
    duration_outlier_count /
    len(watch_history_df) * 100,
    2)

print(
    "Sessions longer than content:",
    duration_outlier_count)

print(
    "Affected percentage:",
    duration_outlier_percentage,
    "%")

duration_outliers_df = watch_history_df.loc[
    duration_outlier_mask,
    [
        "watch_id",
        "subscriber_id",
        "title_id",
        "content_duration_min",
        "watch_duration_min"]].copy()

duration_outliers_df["excess_minutes"] = (
    duration_outliers_df["watch_duration_min"] -
    duration_outliers_df["content_duration_min"])

duration_outliers_df.head(10)

Sessions longer than content: 0
Affected percentage: 0.0 %


,watch_id,subscriber_id,title_id,content_duration_min,watch_duration_min,excess_minutes


Sessions longer than the content: 0
Affected percentage: 0.0%
All 650,000 watch-history rows pass the duration rule.

In [12]:
#Checking referential integrity

valid_subscriber_ids = set(
    all_dfs["subscribers"]["subscriber_id"])

valid_title_ids = set(
    all_dfs["titles"]["title_id"])

invalid_subscriber_mask = (
    ~watch_history_df["subscriber_id"]
    .isin(valid_subscriber_ids))

invalid_title_mask = (
    ~watch_history_df["title_id"]
    .isin(valid_title_ids))

integrity_summary_df = pd.DataFrame({
    "Check": [
        "Invalid subscriber_id",
        "Invalid title_id"],
    "Broken Rows": [
        invalid_subscriber_mask.sum(),
        invalid_title_mask.sum()],
    "Broken Percentage": [
        round(
            invalid_subscriber_mask.mean() * 100,
            2),
        round(
            invalid_title_mask.mean() * 100,
            2)],
    "Unique Broken IDs": [
        watch_history_df.loc[
            invalid_subscriber_mask,
            "subscriber_id"
        ].nunique(),
        watch_history_df.loc[
            invalid_title_mask,
            "title_id"
        ].nunique()]})

integrity_summary_df

,Check,Broken Rows,Broken Percentage,Unique Broken IDs
0,Invalid subscriber_id,0,0.0,0
1,Invalid title_id,0,0.0,0


Invalid subscriber_id: 0
Invalid title_id: 0
Referential integrity is fully maintained in watch_history

In [13]:
#Specific Check 1 — Churn date after signup date

subscribers_df = all_dfs["subscribers"]

churned_mask = subscribers_df["churn_date"].notna()

invalid_churn_order_mask = (
    churned_mask &
    (
        subscribers_df["churn_date"] <=
        subscribers_df["signup_date"]
    )
)

total_churned = churned_mask.sum()

invalid_churn_order_count = (
    invalid_churn_order_mask.sum()
)

invalid_churn_order_percentage = round(
    invalid_churn_order_count /
    total_churned * 100,
    2
)

print("Total churned subscribers:", total_churned)

print(
    "Invalid churn-date order:",
    invalid_churn_order_count
)

print(
    "Percentage of churned subscribers affected:",
    invalid_churn_order_percentage,
    "%"
)

invalid_churn_order_df = subscribers_df.loc[
    invalid_churn_order_mask,
    [
        "subscriber_id",
        "signup_date",
        "churn_date",
        "is_active"
    ]
]

invalid_churn_order_df.head(10)

Total churned subscribers: 3801
Invalid churn-date order: 0
Percentage of churned subscribers affected: 0.0 %


,subscriber_id,signup_date,churn_date,is_active


Specific Check 1 passed:

Total churned subscribers: 3,801
Invalid date order: 0
All churn dates occur after signup dates.

In [14]:
#Specific Check 2 — Active subscribers with a churn date


active_mask = (
    subscribers_df["is_active"] == True
)

active_with_churn_mask = (
    active_mask &
    subscribers_df["churn_date"].notna()
)

total_active = active_mask.sum()

active_with_churn_count = (
    active_with_churn_mask.sum()
)

active_with_churn_percentage = round(
    active_with_churn_count /
    total_active * 100,
    2
)

print("Total active subscribers:", total_active)

print(
    "Active subscribers with a churn date:",
    active_with_churn_count
)

print(
    "Percentage of active subscribers affected:",
    active_with_churn_percentage,
    "%"
)

active_churn_mismatch_df = subscribers_df.loc[
    active_with_churn_mask,
    [
        "subscriber_id",
        "signup_date",
        "churn_date",
        "is_active"
    ]
]

active_churn_mismatch_df.head(10)

Total active subscribers: 11199
Active subscribers with a churn date: 0
Percentage of active subscribers affected: 0.0 %


,subscriber_id,signup_date,churn_date,is_active


Specific Check 2 passed:

Active subscribers: 11,199
Active subscribers with a churn date: 0
Therefore, the 11,199 missing churn_date values are expected—not data-quality errors.

Specific Check 3 — Validate completion_pct

Expected completion %=
watch (duration/content duration)*100
	

In [15]:
completion_check_df = watch_history_df[
    [
        "watch_id",
        "content_duration_min",
        "watch_duration_min",
        "completion_pct"
    ]
].copy()

completion_check_df["calculated_completion_pct"] = (
    completion_check_df["watch_duration_min"] /
    completion_check_df["content_duration_min"] *
    100
)

completion_check_df["difference"] = (
    completion_check_df["completion_pct"] -
    completion_check_df["calculated_completion_pct"]
).abs()

tolerance = 0.1

completion_mismatch_mask = (
    completion_check_df["difference"] > tolerance
)

completion_mismatch_count = (
    completion_mismatch_mask.sum()
)

completion_mismatch_percentage = round(
    completion_mismatch_count /
    len(completion_check_df) * 100,
    2
)

print(
    "Completion percentage mismatches:",
    completion_mismatch_count
)

print(
    "Affected percentage:",
    completion_mismatch_percentage,
    "%"
)

completion_check_df.loc[
    completion_mismatch_mask
].head(10)

Completion percentage mismatches: 1845
Affected percentage: 0.28 %


,watch_id,content_duration_min,watch_duration_min,completion_pct,calculated_completion_pct,difference
14,15,70,35.7,51.1,51.000000,0.100000
66,67,61,56.3,92.4,92.295082,0.104918
593,594,64,48.7,76.2,76.093750,0.106250
790,791,74,68.6,92.6,92.702703,0.102703
1311,1312,89,8.1,9.0,9.101124,0.101124
1472,1473,75,43.8,58.5,58.400000,0.100000
2098,2099,75,43.8,58.3,58.400000,0.100000
2247,2248,82,73.8,90.1,90.000000,0.100000
3315,3316,61,53.3,87.5,87.377049,0.122951
3748,3749,63,47.2,74.8,74.920635,0.120635


In [16]:
#Checking assess rounding differences

difference_summary = (
    completion_check_df["difference"]
    .describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    )
    .round(4)
)

difference_summary

count    650000.0000
mean          0.0294
std           0.0211
min           0.0000
50%           0.0265
90%           0.0579
95%           0.0696
99%           0.0885
max           0.1556
Name: difference, dtype: float64

In [17]:
tolerances = [0.1, 0.5, 1.0]

tolerance_summary = []

for value in tolerances:
    mismatch_count = (
        completion_check_df["difference"] > value
    ).sum()

    tolerance_summary.append({
        "Tolerance": value,
        "Mismatch Count": mismatch_count,
        "Mismatch Percentage": round(
            mismatch_count /
            len(completion_check_df) * 100,
            4
        )
    })

tolerance_summary_df = pd.DataFrame(
    tolerance_summary
)

tolerance_summary_df

,Tolerance,Mismatch Count,Mismatch Percentage
0,0.1,1845,0.2838
1,0.5,0,0.0000
2,1.0,0,0.0000


#this confirms our interpretation:

At tolerance 0.1: 1,845 apparent mismatches due to rounding
At tolerance 0.5: 0 mismatches
At tolerance 1.0: 0 mismatches

In [18]:
#Specific Check 4 — Validate sentiment values


reviews_df = all_dfs["reviews"]

allowed_sentiments = [
    "Positive",
    "Neutral",
    "Negative"
]

invalid_sentiment_mask = (
    ~reviews_df["sentiment"].isin(
        allowed_sentiments
    )
)

print(
    "Invalid sentiment rows:",
    invalid_sentiment_mask.sum()
)

print(
    "Affected percentage:",
    round(
        invalid_sentiment_mask.mean() * 100,
        2
    ),
    "%"
)

sentiment_counts_df = (
    reviews_df["sentiment"]
    .value_counts(dropna=False)
    .rename_axis("Sentiment")
    .reset_index(name="Row Count")
)

sentiment_counts_df

Invalid sentiment rows: 0
Affected percentage: 0.0 %


,Sentiment,Row Count
0,Positive,75007
1,Neutral,22759
2,Negative,12234


Specific Check 4 passed:

Positive: 75,007
Neutral: 22,759
Negative: 12,234
Invalid sentiment values: 0

In [19]:
#Specific Check 5 — Validate watchlist links

watchlist_df = all_dfs["watchlist"]

invalid_watchlist_subscriber_mask = (
    ~watchlist_df["subscriber_id"]
    .isin(valid_subscriber_ids)
)

invalid_watchlist_title_mask = (
    ~watchlist_df["title_id"]
    .isin(valid_title_ids)
)

watchlist_integrity_df = pd.DataFrame({
    "Check": [
        "Invalid subscriber_id",
        "Invalid title_id"
    ],
    "Broken Rows": [
        invalid_watchlist_subscriber_mask.sum(),
        invalid_watchlist_title_mask.sum()
    ],
    "Broken Percentage": [
        round(
            invalid_watchlist_subscriber_mask.mean()
            * 100,
            2
        ),
        round(
            invalid_watchlist_title_mask.mean()
            * 100,
            2
        )
    ],
    "Unique Broken IDs": [
        watchlist_df.loc[
            invalid_watchlist_subscriber_mask,
            "subscriber_id"
        ].nunique(),
        watchlist_df.loc[
            invalid_watchlist_title_mask,
            "title_id"
        ].nunique()
    ]
})

watchlist_integrity_df

,Check,Broken Rows,Broken Percentage,Unique Broken IDs
0,Invalid subscriber_id,0,0.0,0
1,Invalid title_id,0,0.0,0


Specific Check 5 passed:

Invalid watchlist subscriber IDs: 0
Invalid watchlist title IDs: 0
All 65,000 watchlist entries link to valid parent records.

In [20]:
#Specific Check 6 — Active subscribers with short tenure


short_tenure_threshold = 3

short_tenure_active_mask = (
    (subscribers_df["is_active"] == True) &
    (
        subscribers_df["tenure_months"] <=
        short_tenure_threshold
    )
)

short_tenure_active_count = (
    short_tenure_active_mask.sum()
)

short_tenure_active_percentage = round(
    short_tenure_active_count /
    total_active * 100,
    2
)

print(
    "Short-tenure threshold:",
    short_tenure_threshold,
    "months"
)

print(
    "Active subscribers with short tenure:",
    short_tenure_active_count
)

print(
    "Percentage of active subscribers:",
    short_tenure_active_percentage,
    "%"
)

short_tenure_active_df = subscribers_df.loc[
    short_tenure_active_mask,
    [
        "subscriber_id",
        "signup_date",
        "tenure_months",
        "plan_type",
        "is_active"
    ]
].sort_values("tenure_months")

short_tenure_active_df.head(10)


Short-tenure threshold: 3 months
Active subscribers with short tenure: 640
Percentage of active subscribers: 5.71 %


,subscriber_id,signup_date,tenure_months,plan_type,is_active
14943,SUB114943,2026-04-29,1,Premium,True
14941,SUB114941,2026-05-13,1,Premium,True
14918,SUB114918,2026-04-22,1,Basic with Ads,True
14901,SUB114901,2026-05-19,1,Premium,True
14857,SUB114857,2026-05-01,1,Standard,True
358,SUB100358,2026-04-24,1,Premium,True
326,SUB100326,2026-04-30,1,Premium,True
279,SUB100279,2026-05-31,1,Standard,True
277,SUB100277,2026-04-05,1,Basic with Ads,True
270,SUB100270,2026-05-05,1,Premium,True


The short-tenure check ran successfully:

Threshold: 3 months or less
Active subscribers affected: 5.71%
This is informational and not a data-quality error, as stated in the PRD.

In [21]:
short_tenure_active_count

np.int64(640)

640 active subscribers have tenure of three months or less, representing 5.71% of active subscribers.

In [22]:
#Final missing-value validation
titles_df = all_dfs["titles"]

missing_license_mask = (
    titles_df["license_expiry"].isna()
)

original_missing_expiry = (
    missing_license_mask &
    (titles_df["is_original"] == True)
).sum()

licensed_missing_expiry = (
    missing_license_mask &
    (titles_df["is_original"] == False)
).sum()

license_missing_validation_df = pd.DataFrame({
    "Check": [
        "Original titles with missing expiry",
        "Non-original titles with missing expiry"
    ],
    "Count": [
        original_missing_expiry,
        licensed_missing_expiry
    ]
})

license_missing_validation_df

,Check,Count
0,Original titles with missing expiry,1966
1,Non-original titles with missing expiry,0


# StreamFlix Phase 1 — Data Quality Report

## Objective

The purpose of Phase 1 was to profile, clean, and validate the six StreamFlix datasets before conducting further analysis. The assessment covered 979,000 records across ratings, reviews, subscribers, titles, watch history, and watchlist data.

## Dataset Profile

| Dataset           |    Rows | Columns |
| ----------------- | ------: | ------: |
| ratings.csv       | 130,000 |       5 |
| reviews.csv       | 110,000 |       7 |
| subscribers.csv   |  15,000 |      14 |
| titles.csv        |   9,000 |      21 |
| watch_history.csv | 650,000 |      10 |
| watchlist.csv     |  65,000 |       5 |

## Key Findings

**Missing values:** Only two columns contained missing values. `churn_date` had 11,199 missing values (74.66%), all belonging to active subscribers; therefore, these blanks are expected. `license_expiry` had 1,966 missing values (21.84%), all belonging to StreamFlix-original titles; therefore, these blanks are also valid. No imputation or deletion was required.

**Duplicates and keys:** No exact duplicate rows were found in any dataset. All primary identifiers—`rating_id`, `review_id`, `subscriber_id`, `title_id`, `watch_id`, and `watchlist_id`—were unique.

**Data types:** Eight date columns were initially stored as text. These were converted to `datetime64[ns]` using the `YYYY-MM-DD` format. No invalid date values were found during conversion. Duration and price columns were already numeric.

**Watch-duration validation:** No viewing sessions had a `watch_duration_min` greater than `content_duration_min`. Therefore, all 650,000 watch-history records passed the duration rule.

**Referential integrity:** Every `subscriber_id` and `title_id` in both `watch_history.csv` and `watchlist.csv` matched valid records in their respective parent tables. No orphan records were found.

## Specific Business-Rule Checks

* All 3,801 churned subscribers had a `churn_date` later than their `signup_date`.
* All 11,199 active subscribers had a blank `churn_date`; no active/churn-date mismatches were found.
* The stored `completion_pct` values were compared with `watch_duration_min ÷ content_duration_min × 100`. The maximum difference was 0.1556 percentage points. Using a documented rounding tolerance of 0.5 percentage points, no mismatches were found.
* Review sentiment contained only the permitted categories: Positive (75,007), Neutral (22,759), and Negative (12,234).
* A short-tenure threshold of three months or less was used. There were 640 active short-tenure subscribers, representing 5.71% of active subscribers. As stated in the requirements, this is informational and not a data-quality error.

## Conclusion

The StreamFlix datasets passed all required Phase 1 quality checks. No duplicate keys, invalid dates, impossible watch durations, broken relationships, invalid sentiment values, or business-rule violations were identified. The observed missing values are structurally expected and should remain unchanged. Based on these checks, the data is suitable for the next phase of analysis.


In [23]:
# Create clean copies of all Phase 1 datasets for Phase 3


clean_dfs = {name: df.copy() for name, df in all_dfs.items()}

print("Clean datasets prepared for Phase 3:")
print(list(clean_dfs.keys()))

Clean datasets prepared for Phase 3:
['ratings', 'reviews', 'subscribers', 'titles', 'watch_history', 'watchlist']


In [24]:
#checking row count of the cleaned dataset

for name, df in clean_dfs.items():
    print(name, ":", df.shape)


ratings : (130000, 5)
reviews : (110000, 7)
subscribers : (15000, 14)
titles : (9000, 21)
watch_history : (650000, 11)
watchlist : (65000, 5)


In [25]:
#KPI Important table are

print("Subscribers:", clean_dfs["subscribers"].shape)
print("Watch History:", clean_dfs["watch_history"].shape)
print("Titles:", clean_dfs["titles"].shape)

Subscribers: (15000, 14)
Watch History: (650000, 11)
Titles: (9000, 21)


In [27]:
# Saving Phase 1 cleaned datasets for Phase 3

import os

phase3_data_path = "phase3_clean_data"

os.makedirs(phase3_data_path, exist_ok=True)

for name, df in clean_dfs.items():
    df.to_csv(
        os.path.join(phase3_data_path, f"{name}_clean.csv"),
        index=False)

print("Phase 3 clean datasets saved successfully.")

Phase 3 clean datasets saved successfully.
